In [11]:
import numpy as np
import scipy as sp
from scipy import special
from scipy.interpolate import interp1d
import sys, os

os.chdir('/project/chihway/junzhou/Project-Roman-Y1/cocoa/Cocoa/projects/roman_y1/')

#Private functions
import importlib
import utils
importlib.reload(utils)
from utils import binning,comoving_distance
#----------------------------------------------------------------------------------
#----------Definition Section: you should modify and check this part---------------
#----------------------------------------------------------------------------------
output_filename = 'roman_y1_SRDlike.mask'
SAVE_FLAG = True

nsrcs = 5
nlens = 5
vmin  = 2.5
vmax  = 250.0
Ntheta= 20
ggl_exclude = []

#Cosmic shear scale cut: add baryon pollution or set a hard cut
#If you choose a hard cut, set a multipole here or None if you dont
shear_ellmax = 3000 #multipole
#If you choose a baryon budget cut, set a number here for each combo of 
#cosmic shear or None if you dont
shear_budget = None #DEFAULT in real: 0.025
dv_pure_filepath = './chains/roman_des_evaluate/roman.modelvector'
dv_pollution_filepath = './chains/roman_des_pollution_evaluate/roman.modelvector'
cov_filepath = './data/cov_roman_des'

gammat_Rmin  = 1000 # Mpc/h # DEFAULT 6Mpc/h
wtheta_Rmin  = 1000 # Mpc/h # DEFAULT 8Mpc/h

#Cosmological parameter
As_1e9 = 2.1
ns     = 0.96605
H0     = 67.32
omegab = 0.0495
omegam = 0.316
mnu    = 0.06
w0pwa  = -1.0
w      = -1.0

#Lens sample, you need this to do a minimum transverse radius cut
nz_filepath = './data/lsst_y1_lens_4cosmolike.nz'
#-----------------------------------------------------------------------------------
#-------------------------------Initialization--------------------------------------
#-----------------------------------------------------------------------------------
Ndv = (nsrcs*(nsrcs+1)+nsrcs*nlens+nlens - len(ggl_exclude))*Ntheta
thetas = binning(vmin, vmax, Ntheta, 'real')

xi_plus_thetamin = None
xi_minus_thetamin = None
dv_pure = None
dv_pollution = None
cov_raw = None

z, chiofz = comoving_distance(As_1e9, ns, H0, omegab, omegam, mnu, w0pwa, w)
f_chi = interp1d(z, chiofz, kind='cubic',fill_value='extrapolate')
try:
    nz = np.loadtxt(nz_filepath)
except:
    raise ErrorValue(f'Can not load nz file {nz_filepath}')

x_medians = []
for i in range(1,nz.shape[1]):
    x, y = nz[:,0], nz[:,i]
    Ns = np.concatenate([[0],sp.integrate.cumulative_trapezoid(y,x)])
    x_median = np.interp(Ns[-1]/2, Ns, x)
    x_medians.append(x_median)
x_medians = np.array(x_medians)

gammat_thetamins = []
wtheta_thetamins = []
for zi in list(x_medians):
    chi_ = f_chi(zi)
    gammat_thetamins.append(gammat_Rmin/chi_*180/np.pi*60) #arcmin
    wtheta_thetamins.append(wtheta_Rmin/chi_*180/np.pi*60) #arcmin
#-----------------------------------------------------------------------------------
#----------------------------Sanity Check-------------------------------------------
#-----------------------------------------------------------------------------------
if (shear_ellmax is None) == (shear_budget is None):
    raise ValueError(f'You must choose one and only one scale cut method for cosmic shear!\
        now you choose both or neither as: \
        shear_ellmax = {shear_ellmax}, \
        shear_budget = {shear_budget}.')
if shear_ellmax is not None:
    j0 = special.jn_zeros(0,1)[0]
    j4 = special.jn_zeros(4,1)[0]
    xi_plus_thetamin = j0/shear_ellmax*180/np.pi*60  #arcmin
    xi_minus_thetamin = j4/shear_ellmax*180/np.pi*60 #arcmin
    
    #xi_plus_thetamin = 1  #arcmin
    #xi_minus_thetamin = 5 #arcmin
    
if shear_budget is not None:
    try:
        dv_pure = np.loadtxt(dv_pure_filepath)[:,1]
        dv_pollution = np.loadtxt(dv_pollution_filepath)[:,1]
        cov_raw = np.loadtxt(cov_filepath)
    except:
        raise ValueError(f'Cannnot load the either {dv_pure_filepath}, \
                            or {dv_pollution_filepath}.')
    cov = np.zeros((Ndv,Ndv))
    for i in range(len(cov_raw)):
        ii,jj = int(cov_raw[i,0]),int(cov_raw[i,1])
        component = cov_raw[i,8]+cov_raw[i,9]
        cov[ii,jj] = component
        cov[jj,ii] = component
#-----------------------------------------------------------------------------------
#--------------------------Operate--------------------------------------------------
#-----------------------------------------------------------------------------------
mask = np.ones(Ndv)
def _probechi2(start,end):
    if shear_budget is not None:
        dv_seg = (dv_pollution - dv_pure)[start:end]
        mask_seg = mask[start:end].astype(bool)
        dv_seg_masked = dv_seg[mask_seg]
        cov_seg = cov[start:end,:][:,start:end]
        cov_seg_masked = cov_seg[mask_seg,:][:,mask_seg]
        invcov_seg_masked = sp.linalg.inv(cov_seg_masked)
        chi2 = dv_seg_masked@invcov_seg_masked@dv_seg_masked
        return chi2
    else:
        return 0

ncombo = 0
ncombo0 = 0
for i in range(nsrcs):
    for j in range(i,nsrcs):
        if shear_ellmax is not None:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>xi_plus_thetamin)
        elif shear_budget is not None:
            dv_seg = (dv_pollution - dv_pure)[ncombo*Ntheta:(ncombo+1)*Ntheta]
            mask_seg = mask[ncombo*Ntheta:(ncombo+1)*Ntheta].astype(bool)
            dv_seg_masked = dv_seg[mask_seg]
            cov_seg = cov[ncombo*Ntheta:(ncombo+1)*Ntheta,:][:,ncombo*Ntheta:(ncombo+1)*Ntheta]
            cov_seg_masked = cov_seg[mask_seg,:][:,mask_seg]
            invcov_seg_masked = sp.linalg.inv(cov_seg_masked)
            chi2 = dv_seg_masked@invcov_seg_masked@dv_seg_masked
            idx_mask=0
            while(chi2>shear_budget):
                mask[ncombo*Ntheta+idx_mask] = 0
                mask_seg = mask[ncombo*Ntheta:(ncombo+1)*Ntheta].astype(bool)
                dv_seg_masked = dv_seg[mask_seg]
                cov_seg = cov[ncombo*Ntheta:(ncombo+1)*Ntheta,:][:,ncombo*Ntheta:(ncombo+1)*Ntheta]
                cov_seg_masked = cov_seg[mask_seg,:][:,mask_seg]
                invcov_seg_masked = sp.linalg.inv(cov_seg_masked)
                chi2 = dv_seg_masked@invcov_seg_masked@dv_seg_masked
                idx_mask+=1
        else:
            raise ValueError('Type of scale cut to Cosmic shear is not specified!')
        chi2 = _probechi2(ncombo*Ntheta, (ncombo+1)*Ntheta)
        print(f'xi+: ni={i}, nj={j}, cut {np.sum(~mask[ncombo*Ntheta:(ncombo+1)*Ntheta].astype(bool))} points, chi2 = {chi2}')
        ncombo += 1
ncombo1 = ncombo
print(f'xi+ completed, {np.sum(mask[ncombo0*Ntheta: ncombo1*Ntheta])}/{len(mask[ncombo0*Ntheta: ncombo1*Ntheta])} left')
print('xi+ completed, index now is ', ncombo)
print(f'total chi2 for xi+ is {_probechi2(ncombo0, ncombo1)}')
for i in range(nsrcs):
    for j in range(i,nsrcs):
        if shear_ellmax is not None:
            mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>xi_minus_thetamin)
        elif shear_budget is not None:
            dv_seg = (dv_pollution - dv_pure)[ncombo*Ntheta:(ncombo+1)*Ntheta]
            mask_seg = mask[ncombo*Ntheta:(ncombo+1)*Ntheta].astype(bool)
            dv_seg_masked = dv_seg[mask_seg]
            cov_seg = cov[ncombo*Ntheta:(ncombo+1)*Ntheta,:][:,ncombo*Ntheta:(ncombo+1)*Ntheta]
            cov_seg_masked = cov_seg[mask_seg,:][:,mask_seg]
            invcov_seg_masked = sp.linalg.inv(cov_seg_masked)
            chi2 = dv_seg_masked@invcov_seg_masked@dv_seg_masked
            idx_mask=0
            while(chi2>shear_budget):
                mask[ncombo*Ntheta+idx_mask] = 0
                mask_seg = mask[ncombo*Ntheta:(ncombo+1)*Ntheta].astype(bool)
                dv_seg_masked = dv_seg[mask_seg]
                cov_seg = cov[ncombo*Ntheta:(ncombo+1)*Ntheta,:][:,ncombo*Ntheta:(ncombo+1)*Ntheta]
                cov_seg_masked = cov_seg[mask_seg,:][:,mask_seg]
                invcov_seg_masked = sp.linalg.inv(cov_seg_masked)
                chi2 = dv_seg_masked@invcov_seg_masked@dv_seg_masked
                idx_mask+=1
        else:
            raise ValueError('Type of scale cut to Cosmic shear is not specified!')
        chi2 = _probechi2(ncombo*Ntheta, (ncombo+1)*Ntheta)
        print(f'xi-: ni={i}, nj={j}, cut {np.sum(~mask[ncombo*Ntheta:(ncombo+1)*Ntheta].astype(bool))} points, chi2 = {chi2}')
        ncombo += 1
ncombo2 = ncombo
print(f'xi- completed, {np.sum(mask[ncombo1*Ntheta: ncombo2*Ntheta])}/{len(mask[ncombo1*Ntheta: ncombo2*Ntheta])} left')
print('xi- completed, index now is ', ncombo)
print(f'total chi2 for xi- is {_probechi2(ncombo1, ncombo2)}')
for i in range(nlens):
    for j in range(nsrcs):
        if [i,j] in ggl_exclude:
            continue
        mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>gammat_thetamins[i])
        chi2 = _probechi2(ncombo*Ntheta, (ncombo+1)*Ntheta)
        print(f'gammat: ni={i}, nj={j}, cut {np.sum(thetas<gammat_thetamins[i])} points, chi2 = {chi2}')
        ncombo += 1
ncombo3 = ncombo
print(f'gammat completed, {np.sum(mask[ncombo2*Ntheta: ncombo3*Ntheta])}/{len(mask[ncombo2*Ntheta: ncombo3*Ntheta])} left')
print('gammat completed, index now is ', ncombo)
print(f'total chi2 for gammat is {_probechi2(ncombo2, ncombo3)}')
for i in range(nlens):
    mask[ncombo*Ntheta:(ncombo+1)*Ntheta] = mask[ncombo*Ntheta:(ncombo+1)*Ntheta]*(thetas>wtheta_thetamins[i])
    chi2 = _probechi2(ncombo*Ntheta, (ncombo+1)*Ntheta)
    print(f'wtheta: ni={i}, nj={j}, cut {np.sum(thetas<wtheta_thetamins[i])} points,chi2 = {chi2}')
    ncombo += 1
ncombo4 = ncombo
print(f'wtheta completed, {np.sum(mask[ncombo3*Ntheta: ncombo4*Ntheta])}/{len(mask[ncombo3*Ntheta: ncombo4*Ntheta])} left')
print('wtheta completed, index now is ', ncombo)
print(f'total chi2 for wtheta is {_probechi2(ncombo3, ncombo4)}')
#----------------------------------------------------------------------------------------------
#----------------------------------Posterior---------------------------------------------------
#----------------------------------------------------------------------------------------------
nlen = len(mask)
out = np.zeros((nlen,2))
out[:,0] = np.arange(nlen)
out[:,1] = mask
if SAVE_FLAG:
    np.savetxt(f'./data/{output_filename}', out)


xi+: ni=0, nj=0, cut 0 points, chi2 = 0
xi+: ni=0, nj=1, cut 0 points, chi2 = 0
xi+: ni=0, nj=2, cut 0 points, chi2 = 0
xi+: ni=0, nj=3, cut 0 points, chi2 = 0
xi+: ni=0, nj=4, cut 0 points, chi2 = 0
xi+: ni=1, nj=1, cut 0 points, chi2 = 0
xi+: ni=1, nj=2, cut 0 points, chi2 = 0
xi+: ni=1, nj=3, cut 0 points, chi2 = 0
xi+: ni=1, nj=4, cut 0 points, chi2 = 0
xi+: ni=2, nj=2, cut 0 points, chi2 = 0
xi+: ni=2, nj=3, cut 0 points, chi2 = 0
xi+: ni=2, nj=4, cut 0 points, chi2 = 0
xi+: ni=3, nj=3, cut 0 points, chi2 = 0
xi+: ni=3, nj=4, cut 0 points, chi2 = 0
xi+: ni=4, nj=4, cut 0 points, chi2 = 0
xi+ completed, 300.0/300 left
xi+ completed, index now is  15
total chi2 for xi+ is 0
xi-: ni=0, nj=0, cut 5 points, chi2 = 0
xi-: ni=0, nj=1, cut 5 points, chi2 = 0
xi-: ni=0, nj=2, cut 5 points, chi2 = 0
xi-: ni=0, nj=3, cut 5 points, chi2 = 0
xi-: ni=0, nj=4, cut 5 points, chi2 = 0
xi-: ni=1, nj=1, cut 5 points, chi2 = 0
xi-: ni=1, nj=2, cut 5 points, chi2 = 0
xi-: ni=1, nj=3, cut 5 points, chi